Structure tool as JSON

In [ ]:
first_tools = [
        { 
            "type": "function",
            "function": {
                "name": "search_internet",
                "description": "Get internet search results for real time information",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "search_query": {
                            "type": "string",
                            "description": "The query to search the web for",
                        }
                    },
                    "required": ["search_query"],
                },
            },
        }
    ]

In [18]:
with open("api_key.txt", "r") as file:
    api_key = file.read().strip()

In [50]:
from groq import Groq
import json

client = Groq(api_key=api_key)

messages = [{"role": "user", "content": "What the first song released by M83?"}]

response = client.chat.completions.create(
    model="qwen/qwen3-32b", 
    messages=messages, 
    tools=first_tools
)

In [23]:
def pprint_response(response):
    print("--- Full Response ---\n")
    print(response, "\n")
    
    print("--- Chat Completion Message ---\n")
    print(response.choices[0].message, "\n")
    
    if response.choices[0].message.tool_calls:
        for i in range(0, len(response.choices[0].message.tool_calls)):
            print(f"--- Tool Call {i+1} ---\n")
            print(f"Function: {response.choices[0].message.tool_calls[i].function.name}\n")
            print(f"Arguments: {response.choices[0].message.tool_calls[i].function.arguments}\n")

In [51]:
pprint_response(response)

--- Full Response ---

ChatCompletion(id='chatcmpl-72939912-5e8f-44e4-a559-244870b89b9a', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning='Okay, I need to find out the first song released by M83. Let me start by recalling what I know about M83. They\'re an electronic music band, right? Their lead singer is Anthony Gonzalez. I think they\'ve been around for a while, maybe late 90s or early 2000s. But I\'m not sure about their first release.\n\nWait, sometimes bands start with EPs or demos before full albums. Maybe their first song was part of an EP. I should check their discography. But since I don\'t have access to real-time data, I need to use the provided search_internet function. The user has asked for the first song, so I should search for "first song released by M83" or "M83 debut single". \n\nLet me make sure I use the right search

In [52]:
response.choices[0].message.reasoning

'Okay, I need to find out the first song released by M83. Let me start by recalling what I know about M83. They\'re an electronic music band, right? Their lead singer is Anthony Gonzalez. I think they\'ve been around for a while, maybe late 90s or early 2000s. But I\'m not sure about their first release.\n\nWait, sometimes bands start with EPs or demos before full albums. Maybe their first song was part of an EP. I should check their discography. But since I don\'t have access to real-time data, I need to use the provided search_internet function. The user has asked for the first song, so I should search for "first song released by M83" or "M83 debut single". \n\nLet me make sure I use the right search terms. Maybe "M83 first release" could also work. The search function will return results that can help me find the answer. Once I get the results, I can look for their earliest single or EP. I should verify if the search results mention their first release. Sometimes bands have differen

In [53]:
from duckduckgo_search import DDGS
import requests

def search_internet(search_query: str) -> list:
    results = DDGS().text(str(search_query), max_results=5)
    return results

In [56]:
import inspect

def run_conversation(prompt, tools):
    
    messages = [{"role": "user", "content": prompt}]
    
    print("\nInitial Message: ", messages)
    
    response = client.chat.completions.create(
        model="qwen/qwen3-32b",
        messages=messages,
        tools=tools,
    )
    response_message = response.choices[0].message
    print("\nResponse Message: ", response_message)
    
    tool_calls = response_message.tool_calls
    print("\nTool Calls: ", tool_calls)
    
    if tool_calls:
        
        available_functions = {
            "search_internet": search_internet,
        } 
        messages.append(response_message)
        
        for tool_call in tool_calls:
            function_name = tool_call.function.name
            function_to_call = available_functions[function_name]
            function_args = json.loads(tool_call.function.arguments)
            
            
            sig = inspect.signature(function_to_call)
            call_args = {
                k: function_args.get(k, v.default)
                for k, v in sig.parameters.items()
                if k in function_args or v.default is not inspect.Parameter.empty
            }
            print(f"\nCalling {function_to_call} with arguments {call_args}")
            
            function_response = str(function_to_call(**call_args))
            
            print("\nFunction Response: ", function_response)

            tool_message = {
                    "tool_call_id": tool_call.id,
                    "role": "tool",
                    "name": function_name,
                    "content": function_response,
                }
            print("\nAppending Message: ", tool_message)
            
            messages.append(tool_message)  

        second_response = client.chat.completions.create(
            model="qwen/qwen3-32b",
            messages=messages,
        )  

        print("\nLLM Response: ", second_response)

        print("\n---Formatted LLM Response---")
        print("\n",second_response.choices[0].message.content)
        
        return

prompt = "what the first song released by M83"

run_conversation(prompt, first_tools)


Initial Message:  [{'role': 'user', 'content': 'what the first song released by M83'}]

Response Message:  ChatCompletionMessage(content=None, role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning='Okay, I need to find out the first song released by M83. Let me start by recalling what I know about M83. They\'re a French electronic music band, right? Their lead singer is Anthony Gonzales. I think they\'ve been around since the early 2000s. But I\'m not sure about their debut single.\n\nFirst, I should check their discography. The earliest albums might give clues. Their first album was "Before the Dawn Heals Us" released in 2001. Maybe the first song from that album is the answer? Wait, "10000 Hz Legend" is a track from that album, but was that their first single? Or maybe another track like "M83" itself?\n\nAlternatively, maybe they released an EP or a single before the album. Sometimes bands put out a single first. Let me think—I don\'t remember an EP 

/var/folders/qc/ndjmyl9n63dbcp9th7chxjj80000gn/T/ipykernel_74961/2373405257.py:5: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  results = DDGS().text(str(search_query), max_results=5)



Function Response:  []

Appending Message:  {'tool_call_id': 'srhczrrjw', 'role': 'tool', 'name': 'search_internet', 'content': '[]'}

LLM Response:  ChatCompletion(id='chatcmpl-8c264639-43e0-4589-a3f4-0807c0a518cf', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='<think>\nOkay, since the internet search didn\'t return results, I need to rely on my existing knowledge and any other methods to figure out the answer. I know that M83 is a French band formed by Anthony Gonzales in the early 2000s. Their first album is called "Before the Dawn Heals Us," released in 2001. The album includes tracks like "10000 Hz Legend" and the title track. Sometimes bands release a single before the album, but I\'m not sure about M83\'s case.\n\nWait, I recall reading that their first single was "10000 Hz Legend," but I\'m not certain. Alternatively, maybe their debut single was part of an EP or another release before the first album. Let me think again. 